# StubHub AWS WAF Benchmark

GoNB benchmark: получение aws-waf-token и проверка search-запроса StubHub.

In [13]:
package main

import (
    "fmt"
    "io"
    "net/http"
    "net/url"
    "strings"
    "time"

    "github.com/andreyfesunov/awswaf"
    "github.com/janpfeifer/gonb/gonbui"
)

const (
    userAgent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36"
    domain    = "www.stubhub.com"
)

var bootstrapURLs = []string{
    "https://www.stubhub.com/secure/Search?q=1",
}

type BenchRow struct {
    Iter          int
    Cookie        string
    CookieLen     int
    CookieMs      int64
    SearchStatus  int
    SearchMs      int64
    SearchBodyLen int
    Err           string
}

func shortenToken(token string) string {
    if token == "" {
        return ""
    }
    if len(token) <= 14 {
        return token
    }
    return token[:8] + "..." + token[len(token)-6:]
}

func fetchBootstrap(client *http.Client) (string, string, error) {
    var lastErr error
    for _, u := range bootstrapURLs {
        req, _ := http.NewRequest(http.MethodGet, u, nil)
        req.Header.Set("accept", "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8")

        resp, err := client.Do(req)
        if err != nil {
            lastErr = err
            continue
        }
        b, _ := io.ReadAll(resp.Body)
        _ = resp.Body.Close()
        body := string(b)

        if strings.Contains(body, "gokuProps") {
            return body, u, nil
        }
        lastErr = fmt.Errorf("no gokuProps on %s (status=%d len=%d)", u, resp.StatusCode, len(body))
    }
    if lastErr == nil {
        lastErr = fmt.Errorf("bootstrap failed")
    }
    return "", "", lastErr
}

func fetchChallengeJS(client *http.Client, host string) string {
    req, _ := http.NewRequest(http.MethodGet, "https://"+host+"/challenge.js", nil)
    resp, err := client.Do(req)
    if err != nil {
        return ""
    }
    defer resp.Body.Close()
    if resp.StatusCode < 200 || resp.StatusCode >= 400 {
        return ""
    }
    b, err := io.ReadAll(resp.Body)
    if err != nil {
        return ""
    }
    return string(b)
}

func runIteration(client *http.Client, q string, timeoutSec int, i int) BenchRow {
    row := BenchRow{Iter: i}

    html, _, err := fetchBootstrap(client)
    if err != nil {
        row.Err = "bootstrap: " + err.Error()
        return row
    }

    goku, host, err := awswaf.Extract(html)
    if err != nil {
        row.Err = "extract: " + err.Error()
        return row
    }

    challengeJS := fetchChallengeJS(client, host)

    started := time.Now()
    solver, err := awswaf.NewWaf(host, domain, userAgent, goku, challengeJS, "", timeoutSec)
    if err != nil {
        row.Err = "newwaf: " + err.Error()
        return row
    }

    token, err := solver.Run()
    row.CookieMs = time.Since(started).Milliseconds()
    if err != nil {
        row.Err = "solve: " + err.Error()
        return row
    }

    row.Cookie = shortenToken(token)
    row.CookieLen = len(token)

    searchURL := "https://www.stubhub.com/secure/Search?q=" + url.QueryEscape(q)
    req, _ := http.NewRequest(http.MethodGet, searchURL, nil)
    req.Header.Set("user-agent", userAgent)
    req.Header.Set("cookie", "aws-waf-token="+token)

    t0 := time.Now()
    resp, err := client.Do(req)
    row.SearchMs = time.Since(t0).Milliseconds()
    if err != nil {
        row.Err = "search: " + err.Error()
        return row
    }
    defer resp.Body.Close()

    b, _ := io.ReadAll(resp.Body)
    row.SearchStatus = resp.StatusCode
    row.SearchBodyLen = len(b)
    return row
}

func printCookieTable(rows []BenchRow) {
    var b strings.Builder
    b.WriteString("| iter | aws-waf-token | token_len | cookie_ms |\n")
    b.WriteString("|---:|---|---:|---:|\n")
    for _, r := range rows {
        b.WriteString(fmt.Sprintf("| %d | `%s` | %d | %d |\n", r.Iter, r.Cookie, r.CookieLen, r.CookieMs))
    }
    gonbui.DisplayMarkdown(b.String())
}

func printSearchTable(rows []BenchRow) {
    var b strings.Builder
    b.WriteString("| iter | aws-waf-token | status_code | cookie_ms | search_ms | search_body_len |\n")
    b.WriteString("|---:|---|---:|---:|---:|---:|\n")
    for _, r := range rows {
        b.WriteString(fmt.Sprintf("| %d | `%s` | %d | %d | %d | %d |\n", r.Iter, r.Cookie, r.SearchStatus, r.CookieMs, r.SearchMs, r.SearchBodyLen))
    }
    gonbui.DisplayMarkdown(b.String())
}

func main() {
    iterations := 20
    query := "taylor"
    timeoutSec := 120

    client := &http.Client{Timeout: time.Duration(timeoutSec) * time.Second}
    rows := make([]BenchRow, 0, iterations)

    for i := 1; i <= iterations; i++ {
        row := runIteration(client, query, timeoutSec, i)
        rows = append(rows, row)
    }

    gonbui.DisplayMarkdown("### Cookie table")
    printCookieTable(rows)
    gonbui.DisplayMarkdown("### Search table")
    printSearchTable(rows)
}


### Cookie table

| iter | aws-waf-token | token_len | cookie_ms |
|---:|---|---:|---:|
| 1 | `dd91112f...2s5n4=` | 334 | 261 |
| 2 | `dd91112f.../qWJDh` | 322 | 284 |
| 3 | `dd91112f...jb8gI=` | 334 | 333 |
| 4 | `dd91112f...ERcXo=` | 334 | 256 |
| 5 | `dd91112f...Bnffc=` | 334 | 301 |
| 6 | `dd91112f...n5C/U=` | 334 | 324 |
| 7 | `dd91112f...y7OGw=` | 334 | 301 |
| 8 | `dd91112f...ZeV+EU` | 322 | 255 |
| 9 | `dd91112f...0yK7E=` | 334 | 258 |
| 10 | `dd91112f...sxlA0=` | 334 | 289 |
| 11 | `dd91112f...b0cfc=` | 334 | 266 |
| 12 | `dd91112f...hPxwo=` | 334 | 364 |
| 13 | `dd91112f...qVCNw=` | 334 | 269 |
| 14 | `dd91112f...F/Bzk=` | 334 | 351 |
| 15 | `dd91112f...gFTPk=` | 334 | 303 |
| 16 | `dd91112f...4rTccR` | 322 | 266 |
| 17 | `dd91112f...kpTUIm` | 322 | 310 |
| 18 | `dd91112f...0JSyo=` | 334 | 268 |
| 19 | `dd91112f...pY4hg=` | 334 | 322 |
| 20 | `dd91112f...Zeu+I=` | 334 | 286 |


### Search table

| iter | aws-waf-token | status_code | cookie_ms | search_ms | search_body_len |
|---:|---|---:|---:|---:|---:|
| 1 | `dd91112f...2s5n4=` | 200 | 261 | 291 | 233291 |
| 2 | `dd91112f.../qWJDh` | 200 | 284 | 286 | 225977 |
| 3 | `dd91112f...jb8gI=` | 200 | 333 | 250 | 233255 |
| 4 | `dd91112f...ERcXo=` | 200 | 256 | 347 | 233255 |
| 5 | `dd91112f...Bnffc=` | 200 | 301 | 260 | 233255 |
| 6 | `dd91112f...n5C/U=` | 200 | 324 | 281 | 225976 |
| 7 | `dd91112f...y7OGw=` | 200 | 301 | 260 | 204008 |
| 8 | `dd91112f...ZeV+EU` | 200 | 255 | 279 | 233255 |
| 9 | `dd91112f...0yK7E=` | 200 | 258 | 270 | 203996 |
| 10 | `dd91112f...sxlA0=` | 200 | 289 | 250 | 233261 |
| 11 | `dd91112f...b0cfc=` | 200 | 266 | 280 | 233255 |
| 12 | `dd91112f...hPxwo=` | 200 | 364 | 314 | 233261 |
| 13 | `dd91112f...qVCNw=` | 200 | 269 | 254 | 225940 |
| 14 | `dd91112f...F/Bzk=` | 200 | 351 | 257 | 233255 |
| 15 | `dd91112f...gFTPk=` | 200 | 303 | 288 | 208866 |
| 16 | `dd91112f...4rTccR` | 200 | 266 | 279 | 203997 |
| 17 | `dd91112f...kpTUIm` | 200 | 310 | 264 | 223533 |
| 18 | `dd91112f...0JSyo=` | 200 | 268 | 246 | 211316 |
| 19 | `dd91112f...pY4hg=` | 200 | 322 | 265 | 233255 |
| 20 | `dd91112f...Zeu+I=` | 200 | 286 | 297 | 233255 |
